# AF2·02 — Axial Attention over the MSA

**Mechanism of the day:** let a network *read* the MSA instead of hand-computing
statistics from it — but do it cheaply, by factorising attention across the grid's
two axes, and let the pair representation whisper to it about which residues look
coupled.

In rung 01 you extracted contacts from an MSA with mutual information and the average
product correction. Those are fixed formulas. AlphaFold replaces them with **learned
attention** over the MSA — and the first thing you hit is a cost problem.

The MSA is a grid of shape `[N_seq, L]`. Naive self-attention over all its entries is
`O((N·L)²)`, and `N` can be thousands. Infeasible. The fix is **axial attention**:
attend along one axis at a time.

- **Row attention** — within each sequence, residues attend to each other across the
  `L` positions. This is where the model reasons about the *structure* of one protein.
  And it is where the **pair representation enters**: a bias term `b_ij` derived from
  `z[i,j]` is added to the attention logits, so residues the pair rep believes are in
  contact get to attend to each other. This single bias is the channel through which
  geometry flows back into sequence reasoning.
- **Column attention** — at each residue position, the `N` sequences attend to each
  other. This is where the model *compares homologs* — the coevolution signal of
  rung 01, now learned.

Alternating the two recovers most of the power of full attention at a fraction of the
cost. You will build both, prove the cost win, and then watch the pair bias do
something concrete: **improve masked-residue recovery, but only at coevolving
positions** — exactly where a contact partner exists to help.

**How to use this notebook:** implement the reps, make the checkpoints pass.
Solutions at the bottom. Trains in ~50s on a laptop CPU.

In [ ]:
import math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0); rng = np.random.default_rng(0)
plt.rcParams['axes.spines.top'] = False; plt.rcParams['axes.spines.right'] = False
BLUE, GREEN, INK = '#2a78d6', '#008300', '#52514e'

Q = 20; L = 48; N = 200          # amino acids, residues, sequences per batch
MASK = Q; VOCAB = Q + 1

# --- the nb01 toy MSA generator (given): conservation + planted contacts + clades ---
BLOCK = list(range(0, 14)); NB = [c for c in range(L) if c not in BLOCK]
rng2 = np.random.default_rng(3)
cand = [(i, j) for i in NB for j in NB if j >= i + 4]; rng2.shuffle(cand)
EDGES, CON = [], set()
for (i, j) in cand[:16]:
    EDGES.append((i, j, rng2.uniform(0.3, 0.6))); CON.add((min(i, j), max(i, j)))
A, B, Cc = NB[2], NB[10], NB[18]
EDGES += [(A, B, 0.9), (B, Cc, 0.9)]; CON |= {(min(A,B),max(A,B)), (min(B,Cc),max(B,Cc))}
PERM = {(i, j): rng.permutation(Q) for (i, j, _) in EDGES}
PROF = np.array([rng.dirichlet(np.ones(Q) * 0.8) for _ in range(L)])
CONS = [{c: int(rng.integers(Q)) for c in BLOCK} for _ in range(4)]
def generate_msa(N):
    m = np.zeros((N, L), int)
    for p in range(L): m[:, p] = rng.choice(Q, size=N, p=PROF[p])
    cl = rng.integers(4, size=N)
    for c in BLOCK:
        for g in range(4):
            h = (cl == g) & (rng.random(N) < 0.9); m[h, c] = CONS[g][c]
    for (i, j, k) in sorted(EDGES, key=lambda e: (e[0], e[1])):
        cp = rng.random(N) < k; m[cp, j] = PERM[(i, j)][m[cp, i]]
    return m
TRUE = np.zeros((L, L), bool)
for (i, j) in CON: TRUE[i, j] = TRUE[j, i] = True
contact_cols = np.array(sorted({i for i, j in CON} | {j for i, j in CON}))

# --- pair representation z, seeded from nb01 coevolution (APC-corrected MI) ---
def coevolution(msa, ps=0.5):
    n, l = msa.shape; oh = np.eye(Q)[msa]; fi = (oh.sum(0) + ps) / (n + Q * ps); M = np.zeros((l, l))
    for i in range(l):
        for j in range(i + 1, l):
            fij = (oh[:, i].T @ oh[:, j] + ps / Q) / (n + ps); fij /= fij.sum()
            M[i, j] = M[j, i] = (fij * np.log(fij / (np.outer(fi[i], fi[j]) + 1e-12) + 1e-12)).sum()
    m = M.copy(); np.fill_diagonal(m, 0); col = m.sum(1, keepdims=True) / (l - 1)
    a = m[~np.eye(l, dtype=bool)].mean(); c = m - (col @ col.T) / a; np.fill_diagonal(c, 0); return c
Z = torch.tensor(coevolution(generate_msa(2000)), dtype=torch.float32)
Z = (Z - Z.mean()) / Z.std()          # our (single-channel) pair representation z[i,j]
print('MSA batch:', N, 'x', L, '| pair rep z:', tuple(Z.shape), '| contacts:', len(CON))

## Part 1 — why axial? the cost argument

Before building anything, make the motivation concrete. Full self-attention over the
whole MSA compares every one of the `N·L` grid cells with every other:

$$
\begin{aligned} \text{full:}&\quad (N\cdot L)^2 \text{ attention entries} \\ \text{axial:}&\quad N\cdot L^2\ (\text{rows}) + L\cdot N^2\ (\text{columns}) = N\cdot L\cdot(L+N) \end{aligned}
$$

The ratio is `N·L / (N + L)` — and for a real MSA (`N` in the thousands, `L` a few
hundred) that is a **hundreds-fold** saving. That is the whole reason AF2 factorises.

### Rep 1 — `axial_speedup(N, L)`
Return the ratio of full-attention entries to axial-attention entries. Confirm it is
large for realistic MSA sizes.

In [ ]:
def axial_speedup(N, L):
    '''(full attention cost) / (axial attention cost).'''
    # YOUR CODE HERE
    # hint: full = (N*L)**2 ; axial = N*L**2 + L*N**2 ; return full/axial
    raise NotImplementedError

# --- checkpoint ---
assert abs(axial_speedup(200, 48) - (200*48)/(200+48)) < 1e-6
big = axial_speedup(4000, 256)
assert big > 100, 'for a realistic MSA the saving should be >100x'
print('axial speedup: toy MSA (200x48) %.0fx  |  realistic (4000x256) %.0fx'
      % (axial_speedup(200, 48), big))

## Part 2 — row attention with a pair bias

Row attention runs *within each sequence*: the `L` residues attend to each other. We
give it the standard multi-head plumbing (projections, reshapes) and you write the
core: scaled dot-product attention **plus a per-head bias** read off the pair
representation.

The bias is the important part. `pair_bias` has shape `[h, L, L]` — one `L×L` bias map
per head, projected from `z[i,j]`. Adding it to the logits *before* the softmax means:
"head `k` should pay extra attention from `i` to `j` whenever the pair rep says so."
That is the entire mechanism by which structure beliefs steer sequence reasoning.

### Rep 2 — `row_attention(q, k, v, pair_bias)`
`q, k, v` are `[N, h, L, d]` (per sequence, per head, over residues). `pair_bias` is
`[h, L, L]`. Return the attention output `[N, h, L, d]`: softmax over the last axis of
`(q·kᵀ)/√d + pair_bias`, applied to `v`. The bias broadcasts over the `N` sequences.

In [ ]:
def row_attention(q, k, v, pair_bias):
    '''Scaled dot-product attention over residues, with an additive per-head pair bias.'''
    # YOUR CODE HERE
    # hint: d = q.size(-1); logits = q @ k.transpose(-2,-1) / sqrt(d) + pair_bias[None]
    #       att = softmax(logits, -1); return att @ v
    raise NotImplementedError

# --- checkpoint ---
N_, h, Ld, d = 5, 4, L, 8
q, k, v = (torch.randn(N_, h, Ld, d) for _ in range(3))
zero_bias = torch.zeros(h, Ld, Ld)
out = row_attention(q, k, v, zero_bias)
assert out.shape == (N_, h, Ld, d), 'wrong shape'
# a strong positive bias from residue 0 to residue 7 must pull 0's output toward v[7]
bias = torch.zeros(h, Ld, Ld); bias[:, 0, 7] = 50.0
out_b = row_attention(q, k, v, bias)
assert torch.allclose(out_b[:, :, 0, :], v[:, :, 7, :], atol=1e-3), 'bias must steer attention'
print('row attention ok — a pair bias provably redirects who attends to whom ✓')

### Rep 3 — `pair_bias_from_z(z, proj)`
Turn the pair representation into that per-head bias. `z` is `[L, L]` (our single
channel; in real AF2 it is `[L, L, c_z]`). `proj` is an `nn.Linear(1, h)`. Return
`[h, L, L]`. This is the literal MSA↔pair coupling — one small linear layer.

In [ ]:
def pair_bias_from_z(z, proj):
    '''Project pair rep z[L,L] -> per-head attention bias [h, L, L] via proj: Linear(1,h).'''
    # YOUR CODE HERE
    # hint: proj(z[..., None]) is [L, L, h]; permute to [h, L, L]
    raise NotImplementedError

# --- checkpoint ---
proj = nn.Linear(1, 4)
b = pair_bias_from_z(Z, proj)
assert b.shape == (4, L, L), 'should be one LxL bias per head'
print('pair-bias projection ok — z[i,j] becomes a %d-head attention bias ✓' % b.shape[0])

## Part 3 — column attention

Column attention runs *across sequences* at a fixed residue: the `N` homologs attend
to each other. No pair bias here — this axis is about comparing sequences, which is
exactly the coevolution comparison of rung 01, now learned and gated. Same core as
row attention, minus the bias.

### Rep 4 — `column_attention(q, k, v)`
`q, k, v` are `[L, h, N, d]` (per residue, per head, over sequences). Plain
scaled dot-product attention over the last-but-one axis (the `N` sequences).
Return `[L, h, N, d]`.

In [ ]:
def column_attention(q, k, v):
    '''Scaled dot-product attention over the sequence axis.'''
    # YOUR CODE HERE
    raise NotImplementedError

# --- checkpoint ---
q, k, v = (torch.randn(L, 4, 12, 8) for _ in range(3))
out = column_attention(q, k, v)
assert out.shape == (L, 4, 12, 8)
# with identical keys, attention is uniform -> output is the mean over sequences
qk = torch.randn(L, 4, 12, 8); vk = torch.randn(L, 4, 12, 8)
kconst = torch.zeros(L, 4, 12, 8)
uni = column_attention(qk, kconst, vk)
assert torch.allclose(uni, vk.mean(2, keepdim=True).expand_as(vk), atol=1e-4), 'uniform check'
print('column attention ok — mixes information across homologous sequences ✓')

## Part 4 — assemble and train

We wrap your two attention cores into gated multi-head modules (AlphaFold gates every
attention output with a sigmoid — it lets the network suppress uninformative updates),
stack a couple of axial blocks, and train on **masked-MSA recovery**: hide 15% of the
residues and predict them. All given — you wrote the parts that matter.

In [ ]:
class RowAttnPairBias(nn.Module):
    def __init__(self, cm, h=4):
        super().__init__(); self.h, self.d = h, cm // h
        self.q = nn.Linear(cm, cm, bias=False); self.k = nn.Linear(cm, cm, bias=False)
        self.v = nn.Linear(cm, cm, bias=False)
        self.bias = nn.Linear(1, h, bias=False)
        self.g = nn.Linear(cm, cm); self.o = nn.Linear(cm, cm)
    def forward(self, m, z):
        Nn, Ll, cm = m.shape
        sp = lambda x: x.view(Nn, Ll, self.h, self.d).permute(0, 2, 1, 3)
        out = row_attention(sp(self.q(m)), sp(self.k(m)), sp(self.v(m)), pair_bias_from_z(z, self.bias))
        out = out.permute(0, 2, 1, 3).reshape(Nn, Ll, cm)
        return self.o(out * torch.sigmoid(self.g(m)))          # gated

class ColAttn(nn.Module):
    def __init__(self, cm, h=4):
        super().__init__(); self.h, self.d = h, cm // h
        self.q = nn.Linear(cm, cm, bias=False); self.k = nn.Linear(cm, cm, bias=False)
        self.v = nn.Linear(cm, cm, bias=False)
        self.g = nn.Linear(cm, cm); self.o = nn.Linear(cm, cm)
    def forward(self, m):
        mt = m.transpose(0, 1); Ll, Nn, cm = mt.shape
        sp = lambda x: x.view(Ll, Nn, self.h, self.d).permute(0, 2, 1, 3)
        out = column_attention(sp(self.q(mt)), sp(self.k(mt)), sp(self.v(mt)))
        out = out.permute(0, 2, 1, 3).reshape(Ll, Nn, cm)
        return (self.o(out * torch.sigmoid(self.g(mt)))).transpose(0, 1)

class MSAStack(nn.Module):
    def __init__(self, cm=48, nblocks=2, use_bias=True):
        super().__init__(); self.use_bias = use_bias
        self.emb = nn.Embedding(VOCAB, cm); self.pos = nn.Embedding(L, cm)
        self.row = nn.ModuleList([RowAttnPairBias(cm) for _ in range(nblocks)])
        self.col = nn.ModuleList([ColAttn(cm) for _ in range(nblocks)])
        self.ln = nn.ModuleList([nn.LayerNorm(cm) for _ in range(2 * nblocks)])
        self.head = nn.Linear(cm, VOCAB)
    def forward(self, x, z):
        m = self.emb(x) + self.pos(torch.arange(L))
        zb = z if self.use_bias else torch.zeros_like(z)
        li = 0
        for r, c in zip(self.row, self.col):
            m = m + r(self.ln[li](m), zb); li += 1
            m = m + c(self.ln[li](m)); li += 1
        return self.head(m)

def mask_batch(msa):
    x = torch.tensor(msa); y = x.clone()
    sel = torch.rand(x.shape) < 0.15; x[sel] = MASK
    return x, y, sel

def train_stack(use_bias, steps=300, seed=1):
    torch.manual_seed(seed)
    net = MSAStack(use_bias=use_bias); opt = torch.optim.AdamW(net.parameters(), lr=3e-3)
    for st in range(steps):
        x, y, sel = mask_batch(generate_msa(N))
        if sel.sum() == 0: continue
        loss = F.cross_entropy(net(x, Z)[sel], y[sel])
        opt.zero_grad(); loss.backward(); opt.step()
    return net

def recovery(net, reps=6):
    '''masked-token accuracy, split by contact vs non-contact columns.'''
    net.eval(); aC = aN = nC = nN = 0
    with torch.no_grad():
        for _ in range(reps):
            x, y, sel = mask_batch(generate_msa(N)); corr = (net(x, Z).argmax(-1) == y) & sel
            for c in range(L):
                if c in contact_cols: aC += corr[:, c].sum().item(); nC += sel[:, c].sum().item()
                else: aN += corr[:, c].sum().item(); nN += sel[:, c].sum().item()
    return aC / max(nC, 1), aN / max(nN, 1)

t0 = time.time()
net_bias = train_stack(use_bias=True)
net_nobias = train_stack(use_bias=False)
print('trained both stacks in %.0fs' % (time.time() - t0))

## Part 5 — the payoff: the pair bias helps, but *only where it should*

We trained two identical stacks — one with the pair bias wired in, one with it forced
to zero — and measure masked-residue recovery, split by whether a column is part of a
planted contact. The prediction: the bias helps at **contact** columns (there is a
coevolving partner for it to point at) and does **nothing** at non-contact columns
(no partner exists). If the bias helped everywhere, it would just be extra capacity;
that it helps *specifically at contacts* proves it is working through the structure.

### Rep 5 — `recovery_gain(with_bias, without_bias)`
Given two `(contact_acc, noncontact_acc)` tuples, return
`(contact_gain, noncontact_gain)` — the accuracy improvement the bias buys at each.

In [ ]:
def recovery_gain(with_bias, without_bias):
    '''(contact_gain, noncontact_gain) from two (contact_acc, noncontact_acc) tuples.'''
    # YOUR CODE HERE
    raise NotImplementedError

# --- checkpoint ---
rb, rn = recovery(net_bias), recovery(net_nobias)
gain_c, gain_n = recovery_gain(rb, rn)
assert gain_c > 0.03, 'the pair bias should clearly help at contact columns'
assert abs(gain_n) < 0.03, 'and should NOT materially change non-contact columns'
print('recovery @ CONTACT cols:    bias %.3f   no-bias %.3f   gain %+.3f' % (rb[0], rn[0], gain_c))
print('recovery @ NONcontact cols: bias %.3f   no-bias %.3f   gain %+.3f' % (rb[1], rn[1], gain_n))
print('\nThe bias helps only where a coevolving partner exists — it works through')
print('the contact structure, exactly as the MSA<->pair coupling is meant to. ✓')

fig, ax = plt.subplots(figsize=(5.4, 3.4))
x = np.arange(2); w = 0.36
ax.bar(x - w/2, [rn[0], rn[1]], w, label='no pair bias', color='#cbd5e1')
ax.bar(x + w/2, [rb[0], rb[1]], w, label='with pair bias', color=BLUE)
ax.set_xticks(x); ax.set_xticklabels(['contact\ncolumns', 'non-contact\ncolumns'])
ax.set_ylabel('masked-residue recovery'); ax.set_title('the pair bias helps only at contacts')
ax.legend(frameon=False, fontsize=9); ax.grid(alpha=.15, axis='y')
plt.show()

In [ ]:
# One more look: for a contact residue, where does row attention point after training?
net_bias.eval()
with torch.no_grad():
    x, y, sel = mask_batch(generate_msa(N))
    m = net_bias.emb(x) + net_bias.pos(torch.arange(L))
    r0 = net_bias.row[0]
    sp = lambda t: t.view(N, L, r0.h, r0.d).permute(0, 2, 1, 3)
    mm = net_bias.ln[0](m)
    q, k = sp(r0.q(mm)), sp(r0.k(mm))
    pb = pair_bias_from_z(Z, r0.bias)
    att = F.softmax(q @ k.transpose(-2, -1) / math.sqrt(r0.d) + pb[None], -1)
    att = att.mean((0, 1)).numpy()      # avg over sequences & heads -> [L, L]

fig, ax = plt.subplots(figsize=(4.6, 4.2))
ax.imshow(att, cmap='viridis')
yi, xi = np.where(np.triu(TRUE))
ax.scatter(xi, yi, s=16, facecolors='none', edgecolors='#ff5555', lw=1.0, label='true contact')
ax.set_xlabel('key residue j'); ax.set_ylabel('query residue i')
ax.set_title('row attention concentrates on true contacts'); ax.legend(frameon=False, fontsize=8)
plt.show()
print('Bright attention lands on the red contact circles — the bias taught the MSA')
print('attention to route information along the contact graph. ✓')

## Reflection — what just transferred

- **Axial attention** makes reading a big MSA affordable: factor the `[N,L]` grid into
  row (residues) and column (sequences) passes, turning `(N·L)²` into `N·L·(N+L)` — a
  hundreds-fold saving at realistic sizes.
- **Row attention carries the pair bias.** One linear layer turns `z[i,j]` into an
  attention bias, and that is the entire channel through which the pair representation
  (structure beliefs) steers how residues share information. You proved it works: the
  bias improves recovery *only at coevolving positions*.
- **Column attention is learned coevolution.** Comparing homologs at a fixed residue is
  exactly rung 01's column comparison, now a trainable, gated operation.
- **Gating** lets the network veto uninformative attention updates — a small robustness
  trick AF2 applies throughout.
- Together these are the MSA-side half of one Evoformer block. The other half updates
  the *pair* representation — and that is where the real magic (and the fix for
  indirect couplings) lives.

**Next rung:** `AF2·03 — The pair representation & triangle operations`. We build the
communication from MSA to pair (the outer-product mean), then the **triangle
multiplicative update** and **triangle attention** — the geometric-consistency engine
that finally distinguishes a direct contact from the indirect coupling that fooled
mutual information in rung 01.

---
Scroll down only after you've done the reps.

## Solutions appendix (peek only after trying)

In [ ]:
def axial_speedup(N, L):
    return (N * L) ** 2 / (N * L ** 2 + L * N ** 2)

def row_attention(q, k, v, pair_bias):
    d = q.size(-1)
    logits = q @ k.transpose(-2, -1) / math.sqrt(d) + pair_bias[None]   # bias broadcasts over N
    return F.softmax(logits, dim=-1) @ v

def pair_bias_from_z(z, proj):
    return proj(z[..., None]).permute(2, 0, 1)      # [L,L,1]->[L,L,h]->[h,L,L]

def column_attention(q, k, v):
    d = q.size(-1)
    return F.softmax(q @ k.transpose(-2, -1) / math.sqrt(d), dim=-1) @ v

def recovery_gain(with_bias, without_bias):
    return with_bias[0] - without_bias[0], with_bias[1] - without_bias[1]

print('reference solutions loaded — re-run the checkpoint cells above')